# referenced_works with year and source

The citation edge list with **publication year and source (journal) attached to both
endpoints**, written to `OpenAlex/output/referenced_works_w_year/`.

| in | |
|---|---|
| `works/works` | `id` (`W…`), `publication_year` — 1,334 partitions, ~396M works |
| `works/primary_locations` | `work_id`, `source_id` (`S…`) — one row per work, 91.2% of works |
| `works/locations` | the same columns, 1.17 rows per work — used only to fill the gap |
| `works/referenced_works` | `work_id`, `referenced_work_id` — ~1.78B edges |

| out | |
|---|---|
| `work_id`, `work_year`, `work_id_source_id` | the citing work |
| `referenced_work_id`, `referenced_work_year`, `referenced_work_id_source_id` | the cited work |

**Why the maps have to be global.** References cross partitions: of the 1,979,784 distinct
`referenced_work_id` values in `part_0000`, only 5,544 are works held in that same partition.
Attaching anything partition by partition would leave ~99.7% of the cited side null. So pass 1
reads year and source for every work in the snapshot, and pass 2 maps against that.

**How the maps fit in memory.** A dict of 396M string keys would not. Both id families are a
letter followed by digits — `W3002427681`, `S4210205742`, largest ~7.4e9 — so `int(id[1:])`
round-trips through int64. The maps become sorted `int64` arrays with `int16` year and `int64`
source alongside, about 7 GB in total, and the lookup is `np.searchsorted`.

**Which location.** `primary_locations` is one row per work and covers 91.2%; `locations`
covers 91.4% but at 1.17 rows per work, so it cannot answer "the source of this work" without
a tie-break. The rule here is primary first, then the first `locations` row for the 0.17% of
works that have a location but no primary one. Works with neither get a null source, which is
the honest answer and is countable afterwards.

**Cost.** Budget a few hours on a compute node and ~40 GB. **Not a login node** — its watchdog
SIGKILLs work this size with no traceback.

In [ ]:
# (1) Setup
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq

sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa

WORKS   = oa.WORKS                                    # renli's works/ tree, read-only
OUT_DIR = f"{oa.OUT}/referenced_works_w_year"
MAPS    = f"{oa.CACHE}/work_year_source_map.npz"      # pass 1's result, built once
os.makedirs(OUT_DIR, exist_ok=True)

COMPRESSION = "zstd"              # ids repeat heavily; dictionary pages compress hard
YEAR_MIN, YEAR_MAX = 1000, 2030   # publication_year runs 1803-2026 with 0.9% null; this
                                  # rejects impossible values only, it does not narrow scope

wp = sorted(glob.glob(f"{WORKS}/works/part_*.parquet"))
pp = sorted(glob.glob(f"{WORKS}/primary_locations/part_*.parquet"))
lp = sorted(glob.glob(f"{WORKS}/locations/part_*.parquet"))
rp = sorted(glob.glob(f"{WORKS}/referenced_works/part_*.parquet"))
print(f"works      : {WORKS}")
print(f"maps       : {MAPS}")
print(f"output     : {OUT_DIR}")
print(f"partitions : works {len(wp)}, primary_locations {len(pp)}, "
      f"locations {len(lp)}, referenced_works {len(rp)}")
assert len(wp) == len(rp) == len(pp) == len(lp), "partition counts should match"

## (2) Pass 1 — year and source for every work

Cached to `cache/work_year_source_map.npz`. It takes a few minutes; everything after it is a
lookup, so it is built once and reloaded on later runs.

Three arrays, all indexed the same way: the sorted work code, its year (`-1` when the work has
no usable `publication_year`), and its source code (`-1` when it has no location).

In [ ]:
# (2) Build (or load) the global maps
def build_maps(force=False):
    if os.path.exists(MAPS) and not force:
        z = np.load(MAPS)
        print(f"maps loaded: {len(z['code']):,} works")
        return z["code"], z["year"], z["source"]

    t0 = time.time()

    # --- work -> year, from works/works (the authoritative set of works) ---
    codes, years = [], []
    for i, f in enumerate(wp):
        d = pq.read_table(f, columns=["id", "publication_year"]).to_pandas()
        c = oa.id_to_code(d["id"])
        y = pd.to_numeric(d["publication_year"], errors="coerce").to_numpy()
        y = np.where(np.isfinite(y) & (y >= YEAR_MIN) & (y <= YEAR_MAX), y, -1)
        ok = c >= 0
        codes.append(c[ok]); years.append(y[ok].astype(np.int16))
        del d, c, y, ok
        if (i + 1) % 300 == 0:
            print(f"  works {i+1}/{len(wp)}  [{time.time()-t0:.0f}s]", flush=True)
        gc.collect()
    code_ = np.concatenate(codes); year_ = np.concatenate(years)
    del codes, years; gc.collect()
    order = np.argsort(code_, kind="stable")
    code_, year_ = code_[order], year_[order]
    del order; gc.collect()
    uniq = np.concatenate(([True], code_[1:] != code_[:-1]))
    code_, year_ = code_[uniq], year_[uniq]
    del uniq; gc.collect()
    n = len(code_)
    print(f"  {n:,} works with a year  [{time.time()-t0:.0f}s]", flush=True)

    # --- work -> source. primary_locations first; locations only fills its gaps. ---
    source_ = np.full(n, -1, dtype=np.int64)

    def fill(paths, label, only_missing):
        got = 0
        for i, f in enumerate(paths):
            d = pq.read_table(f, columns=["work_id", "source_id"]).to_pandas()
            d = d.dropna(subset=["source_id"])
            if only_missing:
                d = d.drop_duplicates("work_id")     # 1.17 rows/work -> keep the first
            wc = oa.id_to_code(d["work_id"])
            # source ids are S+digits; the same codec works once the letter is stripped
            sc = pd.to_numeric(d["source_id"].astype(str).str[1:],
                               errors="coerce").fillna(-1).astype(np.int64).to_numpy()
            keep = (wc >= 0) & (sc >= 0)
            wc, sc = wc[keep], sc[keep]
            j = np.searchsorted(code_, wc)
            j = np.clip(j, 0, n - 1)
            hit = code_[j] == wc
            j, sc = j[hit], sc[hit]
            if only_missing:
                m = source_[j] < 0
                j, sc = j[m], sc[m]
            source_[j] = sc
            got += len(j)
            del d, wc, sc, j, hit
            if (i + 1) % 300 == 0:
                print(f"  {label} {i+1}/{len(paths)}  [{time.time()-t0:.0f}s]", flush=True)
            gc.collect()
        return got

    a = fill(pp, "primary_locations", only_missing=False)
    print(f"  primary_locations set {a:,} sources", flush=True)
    b = fill(lp, "locations (gap fill)", only_missing=True)
    print(f"  locations filled {b:,} more", flush=True)

    np.savez(MAPS, code=code_, year=year_, source=source_)
    print(f"[{time.time()-t0:.0f}s] maps: {n:,} works, "
          f"{(year_ >= 0).mean()*100:.1f}% with a year, "
          f"{(source_ >= 0).mean()*100:.1f}% with a source -> {MAPS} "
          f"({os.path.getsize(MAPS)/1e9:.2f} GB)")
    return code_, year_, source_

CODE, YEAR, SOURCE = build_maps()
print(f"memory: {(CODE.nbytes + YEAR.nbytes + SOURCE.nbytes)/1e9:.2f} GB")

## (3) Pass 2 — attach both years and both sources

One output file per input partition, written to `.tmp` and renamed only after the footer is
down, so an interrupt cannot leave a footerless parquet behind. A partition whose output
already exists and reads is skipped, so a killed run resumes by resubmitting.

In [ ]:
# (3) Map both endpoints and write
def lookup(codes):
    """(year, source_id string) for each work code; NaN / None where unknown."""
    i = np.searchsorted(CODE, codes)
    i = np.clip(i, 0, len(CODE) - 1)
    hit = CODE[i] == codes
    yr = np.full(len(codes), np.nan, dtype=np.float32)
    sc = np.full(len(codes), -1, dtype=np.int64)
    yr[hit] = YEAR[i[hit]]
    sc[hit] = SOURCE[i[hit]]
    yr[yr < 0] = np.nan                       # -1 marks "no usable year"
    src = np.where(sc >= 0, np.char.add("S", sc.astype(str)), None)
    return yr, src


SCHEMA = pa.schema([
    ("work_id", pa.string()), ("work_year", pa.int16()),
    ("work_id_source_id", pa.string()),
    ("referenced_work_id", pa.string()), ("referenced_work_year", pa.int16()),
    ("referenced_work_id_source_id", pa.string()),
])


def do_part(idx):
    dst = f"{OUT_DIR}/part_{idx:04d}.parquet"
    if os.path.exists(dst):
        try:
            if pq.ParquetFile(dst).metadata.num_rows > 0:     # footer read, not a name check
                return 0, True
        except Exception:
            print(f"  [warn] {os.path.basename(dst)} unreadable -> rebuilding")

    d = pq.read_table(rp[idx], columns=["work_id", "referenced_work_id"]).to_pandas()
    yu, su = lookup(oa.id_to_code(d["work_id"]))
    yv, sv = lookup(oa.id_to_code(d["referenced_work_id"]))

    # NaN -> int16 with safe=False silently becomes 0, not null: an unknown year would be
    # recorded as year zero and every downstream age would be wrong by the publication year.
    # Build the values and the null mask separately instead.
    def i16(x):
        m = np.isnan(x)
        return pa.array(np.where(m, 0, x).astype(np.int16), type=pa.int16(), mask=m)

    t = pa.table({
        "work_id": pa.array(d["work_id"].to_numpy(), pa.string()),
        "work_year": i16(yu),
        "work_id_source_id": pa.array(su, pa.string()),
        "referenced_work_id": pa.array(d["referenced_work_id"].to_numpy(), pa.string()),
        "referenced_work_year": i16(yv),
        "referenced_work_id_source_id": pa.array(sv, pa.string()),
    }, schema=SCHEMA)
    n = t.num_rows
    del d, yu, su, yv, sv; gc.collect()

    pq.write_table(t, dst + ".tmp", compression=COMPRESSION)
    os.replace(dst + ".tmp", dst)
    del t; gc.collect()
    return n, False


t0 = time.time()
total = skipped = 0
for i in range(len(rp)):
    n, was_done = do_part(i)
    total += n; skipped += int(was_done)
    if (i + 1) % 50 == 0:
        el = time.time() - t0
        print(f"  {i+1}/{len(rp)}  {total:,} edges  "
              f"[{el:.0f}s, ~{el/(i+1)*(len(rp)-i-1)/60:.0f}m left]", flush=True)
print(f"\n[{time.time()-t0:.0f}s] {total:,} edges written, {skipped} partitions already done")
print(f"-> {OUT_DIR}  "
      f"({sum(os.path.getsize(p) for p in glob.glob(OUT_DIR+'/*.parquet'))/1e9:.1f} GB)")

## (4) Verify

Footers for the counts; a sample of partitions for the null rates and a spot check that the
year and source attached to an id are the ones the source datasets give that id.

In [ ]:
# (4) Verify
outs = sorted(glob.glob(f"{OUT_DIR}/*.parquet"))
n_out = sum(pq.ParquetFile(p).metadata.num_rows for p in outs)
n_src = sum(pq.ParquetFile(p).metadata.num_rows for p in rp)
print(f"partitions : {len(outs)}/{len(rp)}")
print(f"edges      : {n_out:,} written vs {n_src:,} in the source   "
      f"{'OK' if n_out == n_src else 'MISMATCH'}")
print(f"size       : {sum(os.path.getsize(p) for p in outs)/1e9:.1f} GB "
      f"(source {sum(os.path.getsize(p) for p in rp)/1e9:.1f} GB)")

pick = outs[:: max(1, len(outs) // 8)][:8]
d = pd.concat([pq.read_table(p).to_pandas() for p in pick], ignore_index=True)
print(f"\nsampled {len(d):,} edges from {len(pick)} partitions")
for c in ("work_year", "work_id_source_id",
          "referenced_work_year", "referenced_work_id_source_id"):
    print(f"  {c:<32} null {d[c].isna().sum():>10,}  ({d[c].isna().mean()*100:5.2f}%)")

both = d.dropna(subset=["work_year", "referenced_work_year"])
back = both.work_year - both.referenced_work_year
print(f"\n  citing year - cited year: median {back.median():.0f}, "
      f"{(back < 0).mean()*100:.2f}% negative (citing something published later)")
print(f"  self-citation by source: "
      f"{(d.work_id_source_id == d.referenced_work_id_source_id).mean()*100:.2f}% of edges "
      f"stay inside one journal")

# spot check against the source datasets
chk = pq.read_table(wp[0], columns=["id", "publication_year"]).to_pandas().dropna()
ymap = dict(zip(chk["id"], chk["publication_year"].astype(int)))
pl = pq.read_table(pp[0], columns=["work_id", "source_id"]).to_pandas().dropna()
smap = dict(zip(pl["work_id"], pl["source_id"]))
s = d[d.work_id.isin(ymap)].head(200_000)
bad_y = int((s.work_year != s.work_id.map(ymap)).sum())
s2 = d[d.work_id.isin(smap)].head(200_000)
bad_s = int((s2.work_id_source_id != s2.work_id.map(smap)).sum())
print(f"\n  spot check vs works/works        : {len(s):,} edges, {bad_y} year disagreements "
      f"{'OK' if bad_y == 0 else 'FAIL'}")
print(f"  spot check vs primary_locations  : {len(s2):,} edges, {bad_s} source disagreements "
      f"{'OK' if bad_s == 0 else 'FAIL'}")
display(d.head(8))

## (5) What this is for

Each edge now carries, for both ends, when it was published and where. That is enough for the
things the other notebooks rebuild from scratch:

- `work_year - referenced_work_year` is the backward citation age — the C3/C5/C10 windows are
  a filter on it, and the sleeping-beauty histogram is it binned.
- the two `source_id` columns give the **journal pair** of a reference, which is the unit the
  Uzzi atypicality null shuffles. `paper_z_score` currently reconstructs those pairs from the
  CSR plus a separate journal lookup; they are columns here.
- `work_id_source_id == referenced_work_id_source_id` marks a citation that stays inside one
  journal.

Join `sources.csv.gz` on either source column for the journal's name, type and ISSN —
`oa_common.read_sources()` returns it as `source_id, journal, source_type, issn_l`.

In [ ]:
# (5) A worked look — journal pairs and citation age
d2 = d.dropna(subset=["work_year", "referenced_work_year"]).copy()
d2["age"] = d2.work_year - d2.referenced_work_year
d2 = d2[d2.work_year.between(1950, 2026) & d2.age.between(0, 100)]
g = d2.groupby((d2.work_year // 10 * 10).astype(int))["age"].agg(["size", "median", "mean"])
print("backward citation age by decade of the citing work (sampled partitions):\n")
print(g.rename(columns={"size": "edges"}).round(1).to_string())

src = oa.read_sources().set_index("source_id")["journal"]
top = (d.dropna(subset=["referenced_work_id_source_id"])
        .referenced_work_id_source_id.value_counts().head(10))
print("\nmost-cited sources in the sample:")
for sid, cnt in top.items():
    print(f"  {sid:<14}{cnt:>9,}  {src.get(sid, '(name not in sources.csv.gz)')}")